# 03-ALU、加法器与组合逻辑电路

> 从 1 位加法器开始，逐步扩展到 8 位加法器和 ALU，再理解补码减法、溢出、乘除法电路以及 Verilog/FPGA 的硬件描述方式。

## 1. 本篇学习范围

本篇对应课程视频 **29—40**，讲义《自己动手做台计算机》第 **48—60 页附近**。具体视频为：

- 29_ALU概念入门
- 30_半加器的电路搭建
- 31_全加器电路的搭建
- 32_8位的加法器电路搭建
- 33_8位加法器的实现
- 34_上拉电阻和下拉电阻
- 35_计算机的溢出
- 36_计算机的补码表示
- 37_减法电路通过补码加法来实现
- 38_乘除法电路的实现
- 39_Verilog和fpga
- 40_逻辑运算电路

本篇重点是**组合逻辑和算术逻辑单元**。第 41 个视频开始讲计算器和计算机的区别，留给后续篇章。

## 2. ALU 是什么：把运算集中起来

**ALU（Arithmetic Logic Unit，算术逻辑单元）**是处理器中负责运算的部件。它接收操作数和控制信号，根据选择执行算术或逻辑运算，并输出结果。

可以先把 ALU 抽象成一个黑盒：

```text
操作数 A ─┐
          ├─> ALU ──> 结果 F
操作数 B ─┘       └─> 状态信息（如进位、溢出）
             ↑
          控制信号
```

ALU 内部并不是一个“会思考的程序”，而是由大量逻辑门、加法器、选择器和连线组成的组合电路。控制信号决定哪条运算路径被选中。

### 2.1 从 1 位到 N 位

一个多位 ALU 通常由多个相同的 1 位电路并排组成：每一位处理对应的 `Aᵢ`、`Bᵢ`，加法时再把低位产生的进位传给高位。这样可以复用同一个设计来处理 4 位、8 位或更宽的数据。

## 3. 半加器：两个 1 位数相加

半加器（Half Adder）用于计算两个 1 位输入 `A`、`B` 的和。它输出：

- **和位 `S`**：当前位的结果；
- **进位 `C`**：是否产生一个要交给高位的进位。

### 3.1 真值表

| A | B | S（和） | C（进位） |
|---:|---:|---:|---:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 1 | 0 |
| 1 | 0 | 1 | 0 |
| 1 | 1 | 0 | 1 |

### 3.2 逻辑表达式

```text
S = A ⊕ B
C = A · B
```

因此，一个半加器可以由一个 XOR 和一个 AND 组成。`1+1` 的结果在当前位为 0，同时向高位产生进位 1，这正好对应最后一行。

## 4. 全加器：把低位进位也加进来

半加器没有“来自低位的进位输入”。当一个多位数相加时，当前位还必须接收低一位传来的 `Cᵢₙ`，因此需要全加器（Full Adder）。

输入为 `A`、`B`、`Cᵢₙ`，输出为和 `S` 与下一位进位 `Cₒᵤₜ`。

### 4.1 全真值表

| A | B | Cᵢₙ | S | Cₒᵤₜ |
|---:|---:|---:|---:|---:|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 1 | 1 | 0 |
| 0 | 1 | 0 | 1 | 0 |
| 0 | 1 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 | 0 |
| 1 | 0 | 1 | 0 | 1 |
| 1 | 1 | 0 | 0 | 1 |
| 1 | 1 | 1 | 1 | 1 |

### 4.2 表达式

```text
S    = A ⊕ B ⊕ Cᵢₙ
Cₒᵤₜ = A·B + A·Cᵢₙ + B·Cᵢₙ
```

和位在三个输入中“1 的个数为奇数”时为 1；进位在至少两个输入为 1 时为 1。

## 5. 用两个半加器搭建全加器

全加器可以拆成两个半加器和一个 OR 门，便于在 Digital 中逐级搭建：

```text
第一阶段：S₁ = A ⊕ B，C₁ = A·B
第二阶段：S  = S₁ ⊕ Cᵢₙ，C₂ = S₁·Cᵢₙ
最终进位：Cₒᵤₜ = C₁ + C₂
```

连接关系可以理解为：

```text
A、B ──> 半加器 1 ──> S₁ ──> 半加器 2 <── Cᵢₙ
          C₁ ───────────────────────┐
                                    OR ──> Cₒᵤₜ
          C₂ ───────────────────────┘
```

### 5.1 为什么这种拆分有用

设计复杂电路时，不必一次画出完整的门级表达式。先做出可验证的子模块，再把子模块连接起来，更容易仿真、复用和排错。

## 6. 从全加器扩展到 8 位加法器

把 8 个全加器串起来，就能构成 8 位加法器：

```text
A₀、B₀、Cin ─> FA₀ ─> S₀、C₁
A₁、B₁、C₁  ─> FA₁ ─> S₁、C₂
...
A₇、B₇、C₇  ─> FA₇ ─> S₇、Cout
```

这里的 `FAᵢ` 是第 `i` 位全加器。最低位接收外部 `Cin`，每一级的 `Cout` 连接到下一级的 `Cin`。

### 6.1 示例

以 8 位无符号数为例：

```text
  00001101   (13)
+ 00000111   ( 7)
------------
  00010100   (20)
```

从最低位开始计算，进位逐位向高位传递。这种结构常称为**串行进位加法器**（Ripple-Carry Adder）。

### 6.2 仿真检查

至少测试以下情况：全 0、只有最低位为 1、产生一次进位、连续进位，以及最高位产生进位。不要只用一个简单样例判断 8 位电路正确。

## 7. 8 位加法器的实现要点

课程视频 32—33 从“搭建 8 个单元”进一步讲到完整实现。做多位电路时，应把以下信号区分清楚：

| 信号 | 含义 |
|---|---|
| `A[7:0]` | 第一个 8 位操作数 |
| `B[7:0]` | 第二个 8 位操作数 |
| `Cin` | 外部输入进位，通常加法时置 0 |
| `S[7:0]` | 8 位结果 |
| `Cout` | 最高位产生的最终进位 |

### 7.1 进位传播延迟

串行进位的优点是结构直观、容易搭建；缺点是高位必须等待低位进位逐级传递，位数越多，最坏延迟越长。真实处理器会用更复杂的快速进位结构改善速度，但学习阶段先掌握串行进位最重要。

### 7.2 模块化命名

在 Digital 或 HDL 中，建议把 `FA0` 到 `FA7` 清楚命名，并用总线表示 `A[7:0]`、`B[7:0]`。命名清晰可以显著降低“某一位接错”的排查成本。

## 8. 上拉电阻与下拉电阻：给输入一个确定的默认值

数字输入端不能长期悬空。悬空时，输入可能受噪声影响，在 0 和 1 之间不稳定，仿真和实物表现也可能不同。

- **上拉电阻（Pull-up）：**通过电阻把输入连接到电源，默认读取为 1；按键动作可以把它拉到 0。
- **下拉电阻（Pull-down）：**通过电阻把输入连接到地，默认读取为 0；按键动作可以把它拉到 1。

电阻的作用是提供偏置并限制电流，避免按下按键时把电源和地直接短接。实际接线时，要同时确认：默认电平、有效电平、按键触点和输入端的连接方式。

这也是为什么同一个按键既可以设计成“按下为 1”，也可以设计成“按下为 0”：区别在于上拉还是下拉以及软件/逻辑如何解释。

## 9. 溢出：结果超出位宽能表示的范围

固定 N 位后，结果只能保留 N 位。超出的高位会被丢弃，或者通过 `Cout` 暴露出来。**溢出不是“电路坏了”，而是当前位宽无法表示这个数学结果。**

### 9.1 无符号溢出

N 位无符号数范围为 `0` 到 `2ᴺ−1`。如果最高位产生 `Cout=1`，通常表示无符号加法超出了范围。例如 4 位：

```text
1111 (15) + 0001 (1) = 1 0000 (16)
```

4 位结果只保留 `0000`，额外的 `1` 是最终进位。

### 9.2 有符号溢出

N 位补码的范围为 `−2ⁿ⁻¹` 到 `2ⁿ⁻¹−1`。有符号加法中，若两个操作数同号而结果变成异号，就发生溢出：

- 正数 + 正数得到负数；
- 负数 + 负数得到正数。

也可用最高位的进位关系表示：

```text
V = Cₙ₋₁ ⊕ Cₙ
```

最高位的进位 `Cₙ` 和符号位进入的进位 `Cₙ₋₁` 不同，说明有符号溢出。注意：**最终进位不等于有符号溢出。**

## 10. 补码：让减法也能使用加法器

计算机通常用补码表示有符号整数。以 N 位为例：

- 正数的补码就是其普通二进制表示；
- 负数的补码：先写出对应正数，逐位取反，再加 1。

### 10.1 4 位示例

```text
+3 = 0011
取反：1100
加 1：1101
所以 -3 = 1101
```

补码的好处是正负数可以使用同一个加法器运算，不需要单独制造一套“减法器”。

### 10.2 补码运算要注意

符号位是最高位，但它也参与加法。运算完成后丢弃最高位之外的进位，只保留 N 位；是否超出有符号范围，则另外通过溢出规则判断。

## 11. 用补码加法实现减法

减法可以改写为：

```text
A − B = A + (¬B + 1)
```

电路实现时，把 `B` 的每一位取反，同时给最低位加法器输入 `Cin=1`，就可以让同一组全加器完成减法。

### 11.1 示例：5−3

用 4 位表示：

```text
  0101       5
- 0011       3
= 0101 + (1100 + 0001)
= 0101 + 1101
= 1 0010
```

保留 4 位得到 `0010`，即 2。这里最高位多出的进位按固定宽度规则丢弃。

### 11.2 加法/减法共用一个电路

常见设计是用一个控制信号 `SUB`：

- `SUB=0`：`B` 原样输入，最低位 `Cin=0`，执行加法；
- `SUB=1`：每一位 `B` 与 `SUB` 异或实现取反，最低位 `Cin=1`，执行减法。

这样只需增加按位 XOR 和一个控制信号，就能让同一条加法器通路支持加法与减法。

## 12. 乘法和除法电路的基本思路

乘除法比加法复杂，通常由多个基本操作循环完成，而不是为每一个数学结果都布置一条固定线路。

### 12.1 乘法：移位、判断与累加

二进制乘法可以查看乘数的每一位：

1. 当前位为 1 时，把被乘数累加到部分积；
2. 被乘数按位左移；
3. 乘数按位右移；
4. 重复直到所有位处理完。

这就是“移位加法”思路。硬件中可以用寄存器、加法器和控制逻辑逐周期完成。

### 12.2 除法：比较、减法与移位

二进制除法可以反复比较当前部分余数和除数：够减就输出商位 1 并做减法，不够减就输出 0，再继续移位取得下一位。

乘除法电路需要在**硬件规模、执行周期和吞吐量**之间做取舍。学习时先抓住“基本运算组合成复杂运算”的思路，不必把它和单个组合门混为一谈。

## 13. Verilog 和 FPGA：用文字描述硬件

Verilog 是硬件描述语言（HDL），用于描述数字电路的输入输出和内部连接。FPGA 可以把这些描述综合为可配置的硬件逻辑。

### 13.1 半加器的 Verilog 示例

```verilog
module half_adder(
    input  wire A,
    input  wire B,
    output wire S,
    output wire C
);
    assign S = A ^ B;
    assign C = A & B;
endmodule
```

这段代码描述的是连接关系，不是按 CPU 顺序执行的普通程序：`S` 和 `C` 会随着输入变化而变化。

### 13.2 从代码到 FPGA 的基本流程

```text
Verilog 描述 → 仿真验证 → 综合 → 布局布线 → 下载到 FPGA → 板上测试
```

无论用图形化 Digital 还是 Verilog，核心工作都一样：明确输入输出、写出逻辑关系、验证真值表，再扩展到多位模块。

## 14. 逻辑运算电路：按位复用基础门

加法器负责算术运算，ALU 还需要逻辑运算。对于 N 位数据，可以把同一个 1 位逻辑门复制 N 份并行工作：

```text
A[7] AND B[7] → F[7]
A[6] AND B[6] → F[6]
...
A[0] AND B[0] → F[0]
```

常见按位逻辑操作包括：

| 操作 | 每一位的关系 | 用途理解 |
|---|---|---|
| AND | `Fᵢ=Aᵢ·Bᵢ` | 掩码、同时满足 |
| OR | `Fᵢ=Aᵢ+Bᵢ` | 合并置位 |
| XOR | `Fᵢ=Aᵢ⊕Bᵢ` | 翻转、比较不同 |
| NOT | `Fᵢ=¬Aᵢ` | 按位取反 |

如果 ALU 同时准备了算术路径和逻辑路径，就需要选择器根据控制信号选择哪一路作为最终输出。

## 15. ALU 的组合方法：数据通路与控制信号

把前面的模块组合起来，可以得到一个教学版 ALU：

1. `A`、`B` 进入算术模块，产生加法或减法结果；
2. `A`、`B` 同时进入 AND、OR、XOR、NOT 等逻辑模块；
3. 控制信号决定选择哪种运算；
4. 多位结果汇合后输出 `F`，并保留需要的进位/溢出信息。

核心思想是**数据通路负责产生候选结果，控制信号负责选择结果**。这为后续 CPU 中“指令控制 ALU 做什么”打下基础。

### 15.1 调试顺序

建议从小到大验证：半加器 → 全加器 → 1 位加/减单元 → 8 位加法器 → 逻辑运算模块 → ALU。每完成一级，都保存一个可独立测试的模块。

## 16. 实验与排错清单

| 检查项 | 典型错误 | 排查方法 |
|---|---|---|
| 真值表 | 漏掉输入组合 | 按 `2ⁿ` 行数检查 |
| 进位 | `Cin` 没接到下一位 | 从低位沿 `C₁、C₂...` 逐级追踪 |
| 位宽 | 高位被截断却没记录 | 同时观察 `S` 和 `Cout` |
| 减法 | 取反了 B 却忘记 `Cin=1` | 检查 `SUB` 控制线 |
| 补码 | 把符号位当普通无符号位 | 按有符号范围判断溢出 |
| 输入端 | 实物按键悬空 | 使用上拉或下拉电阻 |
| 逻辑选择 | 运算结果正确但输出选错 | 检查多路选择器控制信号 |

仿真时最好设计边界用例：0、最大值、最小负数、产生进位、连续进位、正负相减和溢出。

## 17. 一页复习表

| 知识点 | 关键结论 |
|---|---|
| 半加器 | `S=A⊕B`，`C=A·B` |
| 全加器 | 输入含 `Cin`，输出 `S` 和 `Cout` |
| 8 位加法器 | 8 个全加器串联，低位进位传给高位 |
| 上拉/下拉 | 给悬空输入提供稳定的默认 1/0 |
| 无符号溢出 | 最高位产生最终进位 |
| 有符号溢出 | 同号相加得到异号结果，或 `V=Cₙ₋₁⊕Cₙ` |
| 补码 | 负数=对应正数按位取反再加 1 |
| 减法 | `A−B=A+(¬B+1)` |
| 乘除法 | 通过移位、比较、加法/减法和控制逻辑迭代实现 |
| ALU | 算术路径和逻辑路径由控制信号选择 |
| Verilog/FPGA | 用 HDL 描述硬件，经综合映射到可配置逻辑 |

## 18. 自测题与参考答案

1. 半加器为什么需要 XOR 和 AND 两个门？
2. 全加器的 `Cin` 从哪里来？为什么半加器不能直接替代它？
3. 8 位串行进位加法器的主要延迟来自哪里？
4. 4 位补码中，`0101 + 0011` 的结果是什么？是否发生有符号溢出？
5. 如何使用同一个加法器实现 `A−B`？
6. 上拉电阻和下拉电阻分别提供什么默认电平？
7. 为什么 `Cout=1` 不一定代表有符号溢出？
8. Verilog 描述的半加器与普通软件函数有什么本质区别？

### 参考答案

1. XOR 产生和位，AND 产生两个 1 相加时的进位。
2. `Cin` 来自低一位的 `Cout`；半加器没有进位输入，无法处理多位加法的级联。
3. 每一级要等待前一级的进位，因此进位逐级传播。
4. `0101+0011=1000`。4 位补码中 5+3=8 超出正数上限 7，发生有符号溢出。
5. 将 B 按位取反，并把最低位 `Cin` 置 1，计算 `A+(¬B+1)`。
6. 上拉默认 1，下拉默认 0。
7. `Cout` 是无符号进位；有符号溢出要看同号相加是否变号，或比较最高位前后的进位。
8. Verilog 主要描述并行工作的硬件连接和逻辑关系，输入变化时相关输出会同时响应。

## 19. 与原课程资料对应关系

| 本篇内容 | 视频 | 讲义页码范围 |
|---|---|---:|
| ALU 概念 | 29 | 48 起 |
| 半加器、全加器 | 30—31 | 49—53 附近 |
| 8 位加法器 | 32—33 | 54—56 附近 |
| 上拉电阻、下拉电阻 | 34 | 相关电路页 |
| 溢出与补码 | 35—36 | 相关算术页 |
| 补码实现减法 | 37 | 相关算术页 |
| 乘除法电路 | 38 | 相关算术页 |
| Verilog 与 FPGA | 39 | 相关扩展页 |
| 逻辑运算电路 | 40 | 逻辑运算页 |

**资料来源：**《自己动手做台计算机》课程，百度网盘分享链接：`https://pan.baidu.com/s/1RLiC8gC1sXTu7EgkQs9oHw?pwd=6633`，提取码：`6633`。本篇按视频 29—40 和讲义 ALU 起始部分整理；讲义页码以预览页为准，个别视频操作页使用“附近”标注。